# 01 · Initial exploration of the raw data

**MSU BAA Datathon 2026 — Monetizing Trust**

This notebook inspects the organizer-provided data before cleaning. It checks completeness,
identifiers, documented ranges, consistency between files, distributions, and early relationships.
It does **not** impute, cap, drop, or overwrite raw values, and does not select a monetization strategy.

Sources: `data/raw/all_users_combined.xlsx` (`Start here` and `all_users`), the three supplied CSVs,
and `docs/case/Datathon_2026_Case_Addendum.pdf`, section 2 (data dictionary).
The standalone `user_profiles.csv` was not attached; we verify its fields in the combined workbook.

Run all cells from top to bottom using the project Python environment. Paths work from the
repository root or any directory inside it. Charts and aggregate audit tables are written to
`reports/eda/`; no cleaned dataset is produced. Do not commit correspondence or per-cell change logs.


In [ ]:
from pathlib import Path
import hashlib
import platform
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/raw/all_users_combined.xlsx').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run inside the repository with organizer attachments in data/raw/.')
RAW = ROOT / 'data/raw'
OUT = ROOT / 'reports/eda'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titleweight': 'bold',
                     'axes.labelsize': 10, 'font.size': 10})
COLORS = ['#176B87', '#D97B29', '#5C7D49', '#8254A2']
print('Python:', platform.python_version())
print({p: importlib.metadata.version(p) for p in ['pandas', 'numpy', 'matplotlib', 'openpyxl']})

def save_chart(fig, name):
    fig.savefig(OUT / name, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 1. Inventory and schema

Record source hashes so this analysis can be tied to the exact input files. Read the workbook
without exporting or modifying it. Show aggregate information rather than individual user records.


In [ ]:
inventory = pd.DataFrame([
    {'file': p.name, 'bytes': p.stat().st_size,
     'sha256': hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in sorted(RAW.iterdir()) if p.suffix in {'.csv', '.xlsx'}
])
display(inventory)
inventory.to_csv(OUT / 'source_inventory.csv', index=False)
workbook = pd.ExcelFile(RAW / 'all_users_combined.xlsx', engine='openpyxl')
print('Workbook sheets:', workbook.sheet_names)
df = pd.read_excel(workbook, sheet_name='all_users')
profiles = ['user_id', 'age_group', 'region', 'account_age_months',
            'daily_minutes_spent', 'sessions_per_week', 'content_category_preference',
            'engagement_score', 'influence_score', 'privacy_sensitivity_score',
            'trust_score', 'data_sharing_attitude']
csv_names = ['engagement_behavior.csv', 'monetization_potential.csv', 'risk_and_churn_metrics.csv']
csvs = {name: pd.read_csv(RAW / name) for name in csv_names}
expected_columns = set(profiles).union(*(set(data.columns) for data in csvs.values()))
schema_check = pd.Series({
    'rows': len(df), 'columns': df.shape[1], 'unique_user_ids': df.user_id.nunique(),
    'missing_user_ids': int(df.user_id.isna().sum()),
    'duplicate_user_id_rows_after_first': int(df.user_id.duplicated().sum()),
    'fully_duplicated_rows_after_first': int(df.duplicated().sum()),
    'expected_columns': len(expected_columns),
})
display(schema_check.to_frame('value'))
print('Missing expected fields:', sorted(expected_columns - set(df.columns)))
print('Unexpected fields:', sorted(set(df.columns) - expected_columns))
print('Missing profile fields in workbook:', sorted(set(profiles) - set(df.columns)))
print('Standalone user_profiles.csv present:', (RAW / 'user_profiles.csv').exists())
display(df.dtypes.astype(str).rename('loaded_dtype').to_frame())


## 2. Do the CSVs agree with the workbook?

Compare by `user_id`, never by row position. Report missing/duplicate keys before attempting a
one-to-one match. Numeric cells agree when `np.isclose` with `rtol=1e-9, atol=1e-9` is true;
paired missing values count as equal. Mismatches would need investigation before choosing a source.


In [ ]:
reconciliation = []
for name, other in csvs.items():
    row = {'file': name, 'rows': len(other), 'missing_ids': int(other.user_id.isna().sum()),
           'duplicate_ids': int(other.user_id.duplicated().sum()),
           'ids_only_in_csv': len(set(other.user_id.dropna()) - set(df.user_id.dropna())),
           'ids_only_in_workbook': len(set(df.user_id.dropna()) - set(other.user_id.dropna()))}
    can_compare = (row['missing_ids'] == row['duplicate_ids'] == 0
                   and df.user_id.notna().all() and df.user_id.is_unique)
    if can_compare:
        left = df.set_index('user_id')
        right = other.set_index('user_id')
        ids = left.index.intersection(right.index)
        mismatches = 0
        missing_columns = set(right.columns) - set(left.columns)
        for column in right.columns.intersection(left.columns):
            a, b = left.loc[ids, column], right.loc[ids, column]
            if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
                equal = np.isclose(a.to_numpy(dtype=float), b.to_numpy(dtype=float),
                                   rtol=1e-9, atol=1e-9, equal_nan=True)
            else:
                equal = (a.eq(b) | (a.isna() & b.isna())).to_numpy()
            mismatches += int((~equal).sum())
        row.update({'compared_users': len(ids), 'mismatched_cells': mismatches,
                    'missing_workbook_columns': ', '.join(sorted(missing_columns))})
    else:
        row.update({'compared_users': pd.NA, 'mismatched_cells': pd.NA,
                    'missing_workbook_columns': 'Comparison skipped: invalid identifiers'})
    reconciliation.append(row)
reconciliation = pd.DataFrame(reconciliation)
display(reconciliation)
reconciliation.to_csv(OUT / 'source_reconciliation.csv', index=False)


## 3. Missing values

Missing is different from zero. Keep both unchanged here. Inspect how much data is missing,
which fields are affected, and whether missingness differs by age group. Differences may influence
later imputation or exclusion decisions; they do not establish the cause of missingness.


In [ ]:
missing = pd.DataFrame({'missing_cells': df.isna().sum(), 'observed_cells': df.notna().sum()})
missing['missing_pct'] = 100 * missing.missing_cells / len(df)
missing = missing.sort_values(['missing_cells'], ascending=False)
display(missing.loc[missing.missing_cells.gt(0)])
missing.to_csv(OUT / 'missing_values.csv')
rows_with_missing = int(df.isna().any(axis=1).sum())
print(f'{int(df.isna().sum().sum()):,} missing cells; {rows_with_missing:,} users '
      f'({rows_with_missing / len(df):.2%}) have at least one missing field.')
fig, ax = plt.subplots(figsize=(9, 4))
missing.loc[missing.missing_cells.gt(0), 'missing_pct'].sort_values().plot.barh(ax=ax, color=COLORS[0])
ax.set(xlabel='Missing values (% of all users)', title='Missingness is concentrated in selected fields')
save_chart(fig, '01_missingness.png')
missing_by_age = df.assign(any_missing=df.isna().any(axis=1)).groupby('age_group', dropna=False).agg(
    users=('user_id', 'size'), users_with_missing=('any_missing', 'sum'),
    missing_user_pct=('any_missing', lambda s: 100 * s.mean()))
display(missing_by_age)


## 4. Documented validity checks

The addendum defines scores on 0–100, probabilities on 0–1, and account ages on 1–36 months.
Counts must be nonnegative integers. Dollar values must be nonnegative. Daily minutes cannot
exceed 1,440 minutes per day. Missing cells are counted separately, not as range failures.
No upper bound is invented for activity counts; unusually high values are reviewed in section 5.


In [ ]:
category_rules = {
    'age_group': ['18-24', '25-34', '35-44', '45+'],
    'region': ['North America', 'Europe', 'Asia-Pacific', 'Latin America', 'Other'],
    'content_category_preference': ['Fitness', 'Gaming', 'Fashion', 'Lifestyle', 'Creative'],
    'data_sharing_attitude': ['Restrictive', 'Cautious', 'Neutral', 'Open'],
}
scores = [c for c in df if c.endswith('_score') or c.startswith('trust_loss_')]
probabilities = ['baseline_churn_probability', 'ad_targeting_churn_risk',
                 'data_sale_churn_risk', 'subscription_adoption_probability']
counts = ['account_age_months', 'sessions_per_week', 'posts_per_week',
          'likes_per_week', 'shares_per_week', 'comments_per_week', 'content_views_per_day']
dollars = ['predicted_ad_revenue_per_month', 'max_subscription_price']
numeric_columns = [c for c in df if c not in ['user_id', *category_rules]]
numeric = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
flags = pd.DataFrame(False, index=df.index, columns=df.columns)
issues = []
for column in numeric_columns:
    raw, values = df[column], numeric[column]
    lower, upper = 0, np.inf
    if column in scores: upper = 100
    if column in probabilities: upper = 1
    if column == 'account_age_months': lower, upper = 1, 36
    if column == 'daily_minutes_spent': upper = 1440
    parse_failure = raw.notna() & values.isna()
    nonfinite = values.notna() & ~np.isfinite(values)
    out_of_range = values.notna() & (values.lt(lower) | values.gt(upper))
    noninteger = (values.notna() & values.mod(1).abs().gt(1e-9)
                  if column in counts else pd.Series(False, index=df.index))
    flags[column] = parse_failure | nonfinite | out_of_range | noninteger
    issues.append({'field': column, 'rule': f'[{lower}, {upper}]' + ('; integer' if column in counts else ''),
                   'missing': int(raw.isna().sum()), 'nonnumeric': int(parse_failure.sum()),
                   'nonfinite': int(nonfinite.sum()), 'outside_range': int(out_of_range.sum()),
                   'noninteger': int(noninteger.sum()), 'flagged_cells': int(flags[column].sum())})
for column, allowed in category_rules.items():
    flags[column] = df[column].notna() & ~df[column].isin(allowed)
    issues.append({'field': column, 'rule': 'Documented categories', 'missing': int(df[column].isna().sum()),
                   'nonnumeric': 0, 'nonfinite': 0, 'outside_range': 0, 'noninteger': 0,
                   'flagged_cells': int(flags[column].sum())})
quality = pd.DataFrame(issues).set_index('field')
display(quality.loc[quality.flagged_cells.gt(0)])
print('Unexpected category values:', {c: df.loc[flags[c], c].unique().tolist() for c in category_rules})
print('IDs failing expected U + 6 digits pattern:', int((~df.user_id.astype('string').str.fullmatch(r'U\d{6}', na=False)).sum()))
print('Blank/whitespace-only text cells:', int(df.select_dtypes('object').apply(
    lambda s: s.astype('string').str.fullmatch(r'\s*', na=False)).sum().sum()))
print('Users with at least one validity flag:', int(flags.any(axis=1).sum()))
quality.to_csv(OUT / 'validity_checks.csv')


## 5. Numeric distributions and potential outliers

Use the **raw values** for these summaries. Tukey's 1.5 × IQR rule is a screening tool, not a
definition of bad data. High engagement or revenue may be valid and strategically important.
The overview shows every numeric field. The second chart compares full raw distributions with
a zoom to the 1st–99th percentiles; the zoom is visual only and does not alter the data.


In [ ]:
summary = numeric.describe(percentiles=[.01, .25, .5, .75, .95, .99]).T
q1, q3 = numeric.quantile(.25), numeric.quantile(.75)
iqr = q3 - q1
iqr_flags = numeric.lt(q1 - 1.5 * iqr) | numeric.gt(q3 + 1.5 * iqr)
summary['iqr_flags'] = iqr_flags.sum()
summary['validity_flags'] = flags[numeric_columns].sum()
display(summary)
summary.to_csv(OUT / 'numeric_summary.csv')
fig, axes = plt.subplots(7, 4, figsize=(16, 19))
for ax, column in zip(axes.flat, numeric_columns):
    ax.hist(numeric[column].dropna(), bins=35, color=COLORS[0], alpha=.85)
    ax.set_title(column.replace('_', ' '), fontsize=9)
    ax.tick_params(labelsize=8)
for ax in list(axes.flat)[len(numeric_columns):]: ax.set_visible(False)
fig.suptitle('Raw numeric distributions (all observed values)', fontsize=16)
fig.supxlabel('Value in the field’s documented units'); fig.supylabel('Users')
fig.tight_layout(rect=[.02, .02, 1, .97])
save_chart(fig, '02_numeric_distributions.png')

focus = ['account_age_months', 'daily_minutes_spent', 'sessions_per_week', 'likes_per_week']
fig, axes = plt.subplots(len(focus), 2, figsize=(12, 11))
for row, column in enumerate(focus):
    values = numeric[column].dropna()
    lo, hi = values.quantile([.01, .99])
    for col, shown in enumerate([values, values[values.between(lo, hi)]]):
        axes[row, col].hist(shown, bins=40, color=COLORS[col])
        axes[row, col].set_title(column.replace('_', ' ') + (' — full raw range' if col == 0 else ' — 1st–99th percentile zoom'), fontsize=10)
        axes[row, col].set_ylabel('Users')
fig.tight_layout()
save_chart(fig, '03_outlier_review.png')


## 6. User mix and descriptive segment comparisons

Keep missing categories visible. Means use available observations for each metric, so the
table includes the corresponding observed counts. These summaries use raw records and may
be affected by invalid or extreme activity values. They describe associations, not causal effects.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, column in zip(axes.flat, category_rules):
    counts_by_group = df[column].fillna('(missing)').value_counts()
    (100 * counts_by_group / len(df)).sort_values().plot.barh(ax=ax, color=COLORS[0])
    ax.set(title=column.replace('_', ' '), xlabel='Share of all users (%)')
fig.tight_layout()
save_chart(fig, '04_user_mix.png')
segment_metrics = ['engagement_score', 'privacy_sensitivity_score', 'trust_score',
                   'predicted_ad_revenue_per_month', 'ad_targeting_churn_risk', 'data_sale_churn_risk']
segment_tables = {}
for column in ['content_category_preference', 'data_sharing_attitude']:
    groups = df.assign(segment=df[column].fillna('(missing)')).groupby('segment', dropna=False)
    table = groups.size().rename('users').to_frame()
    table['share_pct'] = 100 * table.users / len(df)
    for metric in segment_metrics:
        table[metric + '_mean'] = groups[metric].mean()
        table[metric + '_n'] = groups[metric].count()
    segment_tables[column] = table
    display(Markdown('**Grouped by ' + column.replace('_', ' ') + '**'))
    display(table)
    table.to_csv(OUT / ('segments_by_' + column + '.csv'))


## 7. Early relationships between value, privacy, and risk

Spearman correlation measures monotonic association using pairwise observed values. It is less
sensitive to extreme magnitudes than Pearson correlation, but does not establish causation.
The correlation chart uses only fields with no documented validity failures. Missing values are
excluded separately for each pair. Keep churn probabilities distinct from trust-loss scores.


In [ ]:
relationship_fields = ['engagement_score', 'influence_score', 'privacy_sensitivity_score',
    'trust_score', 'predicted_ad_revenue_per_month', 'max_subscription_price',
    'data_value_score', 'brand_insight_value_score', 'baseline_churn_probability',
    'ad_targeting_churn_risk', 'data_sale_churn_risk', 'subscription_adoption_probability']
correlation = numeric[relationship_fields].corr(method='spearman', min_periods=100)
present = numeric[relationship_fields].notna().astype('int64')
pair_counts = present.T @ present
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(correlation, cmap='RdBu_r', vmin=-1, vmax=1)
labels = [c.replace('_', ' ') for c in relationship_fields]
ax.set_xticks(range(len(labels)), labels, rotation=55, ha='right', fontsize=8)
ax.set_yticks(range(len(labels)), labels, fontsize=8)
ax.set_title('Spearman correlations: observed pairs in raw data')
fig.colorbar(im, ax=ax, shrink=.75, label='Spearman correlation')
fig.tight_layout()
save_chart(fig, '05_correlations.png')
pairs = pd.DataFrame([
    {'field_a': a, 'field_b': b, 'spearman_r': correlation.loc[a, b],
     'observed_pairs': pair_counts.loc[a, b]}
    for i, a in enumerate(relationship_fields) for b in relationship_fields[i + 1:]
]).dropna(subset=['spearman_r'])
pairs['absolute_r'] = pairs.spearman_r.abs()
display(pairs.sort_values('absolute_r', ascending=False).head(12).drop(columns='absolute_r'))
correlation.to_csv(OUT / 'spearman_correlations.csv')
pair_counts.to_csv(OUT / 'correlation_pair_counts.csv')


## 8. First findings and cleaning decisions to make

The following findings are calculated from this run. The suggestions are a starting point for team
discussion, not cleaning rules already applied. No users have been excluded.


In [ ]:
problem_fields = quality.loc[quality.flagged_cells.gt(0), 'flagged_cells']
profile_available = set(profiles).issubset(df.columns)
csv_agree = (reconciliation.mismatched_cells.eq(0).all()
             and reconciliation.ids_only_in_csv.eq(0).all()
             and reconciliation.ids_only_in_workbook.eq(0).all()
             and reconciliation.missing_workbook_columns.eq('').all())
findings = [
    f'Loaded **{len(df):,} users and {df.shape[1]} fields**; '
    f'**{int(df.user_id.duplicated().sum()):,} duplicate user IDs**.',
    f'All documented profile fields are present in the workbook: **{profile_available}**. '
    'The missing standalone CSV does not prevent initial exploration.',
    f'The three supplied CSVs match the workbook by ID and observed values: **{csv_agree}**.',
    f'**{int(df.isna().sum().sum()):,} cells** are missing across **{rows_with_missing:,} users '
    f'({rows_with_missing / len(df):.2%})**. Do not turn these into zeros automatically.',
    'Documented validity flags: ' + ', '.join(f'**{c}: {int(n)}**' for c, n in problem_fields.items()) + '.',
    f'Likes per week range up to **{numeric.likes_per_week.max():,.0f}**, compared with a '
    f'99th percentile of **{numeric.likes_per_week.quantile(.99):,.1f}**. '
    'Investigate the tail rather than automatically deleting highly active users.',
]
next_steps = [
    'Agree how to handle account ages outside 1–36 months, negative activity, and daily minutes above 1,440. '
    'Consider marking invalid cells missing rather than deleting an entire user.',
    'Choose missing-value rules separately for categories, activity, and revenue. '
    'Compare complete-case, imputed, and conservative revenue scenarios where appropriate.',
    'Review unusually high likes as possible errors or genuine heavy users; test whether capping them changes conclusions.',
    'Implement agreed rules in a separate cleaning step and write its output to data/processed/. '
    'Keep data/raw/ untouched and retain a log of changes.',
    'Repeat key summaries after cleaning, then build the strategy revenue/churn/runway model using the addendum.',
]
summary_text = '# Initial EDA findings\n\n' + '\n'.join('- ' + x for x in findings)
summary_text += '\n\n## Decisions before cleaning\n\n' + '\n'.join('- ' + x for x in next_steps)
summary_text += '\n\nThese are initial descriptive results, not a monetization recommendation.\n'
display(Markdown(summary_text))
(OUT / 'findings.md').write_text(summary_text)

# Confirm this notebook did not modify any source file.
for record in inventory.to_dict('records'):
    assert hashlib.sha256((RAW / record['file']).read_bytes()).hexdigest() == record['sha256']
print('Raw input hashes unchanged. Aggregate tables and figures saved to reports/eda/.')
